# Antarctic sea-ice seasonality

Day of sea-ice advance, day of retreat and season duration,
following Massom et al. (2013):

- **advance** is the first day of the first run of five
  consecutive days at or above 15% concentration;
- **retreat** is the last day at or above 15%;
- **duration** is the difference.

The five-day rule is what separates a real advance from a
transient excursion. A "first day above threshold" definition,
which is easy to write by accident, gives systematically earlier
advance dates.

A sea-ice year runs 15 February to 14 February, so the day
numbers below are relative to 15 February. This needs daily data
(`SIday`); models publishing only `SImon` cannot support it.

### One deliberate difference from the original

The COSIMA notebook computes its advance date as

```python
advanceDate = xr.where(advance == min_days_threshold, advance, False)
advanceDate = advanceDate.argmax(dim='time')
```

where `advance` is the running count of consecutive days above
threshold. That returns the index where the count *reaches* five
— the **fifth** day of the run.

Massom et al. define the day of advance as the day concentration
first exceeds 15% *and then stays above for at least five
consecutive days*: the **first** day of the run. This notebook
follows Massom, so its advance dates are **4 days earlier** than
the original's, and its durations 4 days longer. Retreat is
computed the same way in both and agrees.

Four days is small against a season of a few hundred, but it is
a systematic offset, not noise, and it matters if you compare
these dates against published Massom-based climatologies.

A consequence worth recognising in the output: the shortest
possible duration is 4 days, from a cell whose ice lasts exactly
the five days the rule demands. A minimum of 4 in the printout
below is the rule working, not a truncation.

---

Translated from the COSIMA recipe
[`Sea_Ice_Seasonality_Statistics.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Sea_Ice_Seasonality_Statistics.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
import cartopy.crs as ccrs
import matplotlib.path as mpath
from esmvalcore.preprocessor import extract_region, regrid

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Load the data

In [ ]:
siconc = Dataset(short_name='siconc', project='CMIP6',
                 mip='SIday', exp='historical',
                 dataset='ACCESS-CM2', ensemble='r1i1p1f1',
                 grid='gn', timerange='20000215/20010214')

cube = siconc.load()
cube = extract_region(cube, start_longitude=0., end_longitude=360.,
                      start_latitude=-80., end_latitude=-50.)
cube = regrid(cube, target_grid='1x1', scheme='nearest')
print(cube.summary(shorten=True))

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lon_1d(cube):
    """Representative 1D longitude, also for 2D coordinates."""
    return _coord_points(cube, "longitude")


def lat_2d(cube):
    """2D latitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("latitude")
    if coord.ndim == 2:
        return coord.points
    lon = cube.coord("longitude").points
    return np.broadcast_to(coord.points[:, None], (coord.shape[0], lon.size))


def lon_2d(cube):
    """2D longitude field, broadcasting a 1D coordinate if needed."""
    coord = cube.coord("longitude")
    if coord.ndim == 2:
        return coord.points
    lat = cube.coord("latitude").points
    return np.broadcast_to(coord.points[None, :], (lat.size, coord.shape[0]))


def fraction_from_percent(cube):
    """Sea-ice concentration as a fraction, whatever the input units.

    CMOR ``siconc`` is a percentage but several models publish a
    fraction.  The units are trusted when they are meaningful and the
    data range is used only as a fallback -- masking the fill values
    first, which is the bug that made the Russell sea-ice figures come
    out empty.
    """
    data = masked_data(cube)
    data = np.ma.masked_greater(data, 1.0e15)
    units = str(cube.units).strip().lower()
    if units in ("%", "percent"):
        return data / 100.0
    if units in ("1", "", "unknown", "none"):
        return data / 100.0 if np.ma.max(data) > 5.0 else data
    return data / 100.0 if np.ma.max(data) > 5.0 else data


def seaice_seasonality(concentration, threshold=0.15, consecutive_days=5):
    """Day of advance, day of retreat and season duration.

    ``concentration`` is ``(time, lat, lon)`` daily sea-ice
    concentration as a *fraction*, covering one sea-ice year (15
    February to 14 February).  Following Massom et al. (2013):

    - advance is the first day of the first run of ``consecutive_days``
      days at or above ``threshold``.  Note that the COSIMA notebook
      instead reports the day the run *reaches* five, which is four
      days later; Massom's definition is the first day, and that is
      what this returns;
    - retreat is the last day at or above ``threshold``;
    - duration is the difference.

    Cells that never reach the threshold are masked; cells that never
    drop below it get day 1 and the full year.
    """
    data = np.ma.masked_invalid(np.ma.asarray(concentration, dtype=float))
    above = np.ma.filled(data >= threshold, False)
    n_days = above.shape[0]
    days_above = above.sum(axis=0)

    no_ice = days_above == 0
    no_advance = days_above < consecutive_days
    always_ice = days_above == n_days

    # Length of the current run of "above threshold" days, reset to 0
    # whenever the concentration drops below the threshold.
    run = np.zeros(above.shape, dtype=int)
    counter = np.zeros(above.shape[1:], dtype=int)
    for day in range(n_days):
        counter = np.where(above[day], counter + 1, 0)
        run[day] = counter

    reached = run == consecutive_days
    advance = np.argmax(reached, axis=0).astype(float)
    # argmax returns 0 when nothing is True; those cells are masked by
    # no_advance below.  The advance day is the *start* of the run.
    advance = advance - (consecutive_days - 1) + 1  # 1-based day of year

    reversed_above = above[::-1]
    last_above = n_days - np.argmax(reversed_above, axis=0)
    retreat = last_above.astype(float)

    advance = np.ma.masked_array(advance, mask=no_advance)
    retreat = np.ma.masked_array(retreat, mask=no_ice)
    advance[always_ice] = 1.0
    retreat[always_ice] = float(n_days)
    duration = retreat - advance
    return advance, retreat, duration


def south_polar_axes(figure, position, max_lat=-30.0):
    """Circular South Polar Stereographic axes."""
    import cartopy.crs as ccrs
    import matplotlib.path as mpath

    axes = figure.add_subplot(*position, projection=ccrs.SouthPolarStereo())
    axes.set_extent([-180, 180, -90, max_lat], crs=ccrs.PlateCarree())
    theta = np.linspace(0, 2 * np.pi, 100)
    verts = np.vstack([np.sin(theta), np.cos(theta)]).T
    axes.set_boundary(mpath.Path(verts * 0.5 + [0.5, 0.5]),
                      transform=axes.transAxes)
    return axes

## Compute

`fraction_from_percent` masks fill values *before* deciding
whether the data is a percentage or a fraction. Deciding from
`max(siconc) < 5` with one fill value still present is what
silently empties sea-ice figures.

In [ ]:
concentration = fraction_from_percent(cube)
advance, retreat, duration = seaice_seasonality(
    concentration, threshold=0.15, consecutive_days=5)

print(f'season duration: {np.ma.min(duration):.0f} to '
      f'{np.ma.max(duration):.0f} days')

## Plot

In [ ]:
fields = [(advance, 'Sea ice advance day', 'twilight'),
          (retreat, 'Sea ice retreat day', 'twilight'),
          (duration, 'Sea ice season duration', 'Blues')]

fig = plt.figure(figsize=(16, 5.5))
lon2d, lat2d = lon_2d(cube), lat_2d(cube)

for index, (field, title, cmap) in enumerate(fields):
    ax = south_polar_axes(fig, (1, 3, index + 1), max_lat=-50.0)
    mesh = ax.pcolormesh(lon2d, lat2d, field, cmap=cmap,
                         transform=ccrs.PlateCarree(),
                         shading='auto')
    ax.coastlines(resolution='110m', linewidth=0.5)
    ax.set_title(title)
    fig.colorbar(mesh, ax=ax, orientation='vertical', shrink=0.65,
                 pad=0.06, label='Days since 15 February')

plt.show()

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()